# 2.1 在 keras 中加载 mnist 数据集

In [10]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "0"

from keras.datasets import mnist

(train_images, train_labels), (test_images, test_labels) = mnist.load_data()
# tarin_* 组成训练集，模型从这些数据中进行学习
# test_* 组成测试集，在测试集上对模型进行测试
# 图像被编码成 Numpy 数组，标签是数字数组 0-9，图像和标签一一对应的

## 2.1.1 以下是训练数据

In [12]:
train_images.shape

(60000, 28, 28)

In [8]:
len(train_images)

60000

In [13]:
train_labels

array([5, 0, 4, ..., 5, 6, 8], shape=(60000,), dtype=uint8)

## 2.1.2 以下是测试数据

In [11]:
test_images.shape

(10000, 28, 28)

In [12]:
len(test_images)

10000

In [17]:
test_labels

array([7, 2, 1, ..., 4, 5, 6], shape=(10000,), dtype=uint8)

# 2.2 网络架构

In [3]:
# 将训练数据（train_images, train_labels）输入神经网络
# 网络学习将图像和标签关联在一起
# 网络对 test_images 生成预测，验证这些预测与 test_labels 中的标签是否匹配
from keras import models
from keras import layers

network = models.Sequential()
network.add(layers.Dense(512, activation='relu', input_shape=(28 * 28,)))
network.add(layers.Dense(10, activation='softmax'))
# 神经网络的核心是层（layer），层从输入数据中提取 "表示"
# 深度学习一般都是将简单的 层 链接起来，从而实现渐进式的 数据蒸馏（data distillation）
# 本例中的网络包含 2 个 Dense 层，他们是 密集连接（全连接）的神经层
# 第二层是一个 10 路的 softmax 层，它将返回一个由 10 个概率值（总和为 1）组成的数组
#   Softmax 是一种把“任意实数得分”转换成“概率分布”的函数，主要用于多分类问题。

I0000 00:00:1790702862.661740 1388042 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1351 MB memory:  -> device: 0, name: NVIDIA GeForce MX570, pci bus id: 0000:01:00.0, compute capability: 8.6


# 2.3 编译步骤

In [5]:
# 损失函数（loss function）：网络衡量在训练数据上的性能，即网络如何朝着正确的方向前进
# 优化器（optimizer）：基于训练数据和损失板书来更新网络的机制
# 在训练和测试过程中需要监控的指标（metric）：本例只关心精度，即正确分类数字图片所占的比例
network.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])

# 2.4 数据预处理

In [14]:
# 训练之前，需要对数据进行预处理，将其变成网络要求的形状，并缩放到所有值都在 [0, 1] 区间
# 之前的数据是 uint8 类型的数组，形状为 (60000, 28, 28)，取值区间 [0, 255]
# 将其变换成一个 float32 数组，形状为 (60000, 28 * 28)，取值范围 0~1
train_images = train_images.reshape((60000, 28 * 28))
train_images = train_images.astype('float32') / 255

test_images = test_images.reshape((10000, 28 * 28))
test_images = test_images.astype('float32') / 255

# 还要对标签进行分类编码
from keras.utils import to_categorical

train_labels = to_categorical(train_labels)
test_labels = to_categorical(test_labels)

# 2.5 训练网络

In [17]:
# 训练网络在 keras 中直接调用网络的 fit 方法
network.fit(train_images, train_labels, epochs=5, batch_size=128)

Epoch 1/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8589 - loss: 0.5230
Epoch 2/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8704 - loss: 0.4738
Epoch 3/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8782 - loss: 0.4402
Epoch 4/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8843 - loss: 0.4158
Epoch 5/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8885 - loss: 0.3970


# 2.6 检查模型在测试集上的性能

In [18]:
test_loss, test_acc = network.evaluate(test_images, test_labels)
print(f"test_loss: {test_loss}, test_acc: {test_acc}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.7908 - loss: 49.3811
test_loss: 49.381126403808594, test_acc: 0.7907999753952026
